#### Compare data with difference size  
In P2_featurelevel, all data are from the filtered 4_11 CIC_IoMT

In [3]:
import matplotlib.pyplot as plt
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

X_columns = [
    'Header_Length', 
    # 'Protocol Type', 'Duration', 
    'Rate', 'Srate', 
    # 'Drate',
    # 'fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number',
    # 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number', 'ack_count',
    # 'syn_count', 'fin_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet',
    # 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IGMP', 
    # 'IPv','LLC', 
    'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number',
    'Magnitue', 'Radius', 'Covariance',
    # 'Variance', 'Weight'
]

Y_columns = ['label_L1']

label_L1_mapping = {"MQTT": 0, "Benign": 1, "Recon": 2, "ARP_Spoofing": 3}
label_L2_mapping = {"MQTT-DDoS-Connect_Flood": 4, "MQTT-DDoS-Publish_Flood": 5, 
                    "MQTT-DoS-Connect_Flood": 6, "MQTT-DoS-Publish_Flood": 7,
                    "MQTT-Malformed_Data": 8, "benign": 9, 
                    "Recon-OS_Scan": 10, "Recon-Port_Scan": 11,
                    "arp_spoofing": 12}


# Read the CSV file
# df = pd.read_csv('/home/zyang44/Github/baseline_cicIOT/CIC_IoMT/19classes/filtered_train_l_4_11.csv')
df = pd.read_csv('/home/zyang44/Github/baseline_cicIOT/CIC_IoMT/19classes/filtered_train_s_4_11.csv')
df['label_L1'] = df['label_L1'].map(label_L1_mapping)
df['label_L2'] = df['label_L2'].map(label_L2_mapping)

# Shuffle the dataframe before splitting into training and test sets
df = df.sample(frac=1, random_state=42)
# 90% as training set and 10% as test set
train_size = int(len(df) * 0.9)
train_df, test_df = df.iloc[:train_size, :], df.iloc[train_size:, :]

scaler = StandardScaler()
train_X_scaled = scaler.fit_transform(train_df[X_columns])
test_X_scaled = scaler.transform(test_df[X_columns])
train_y = train_df[Y_columns].values.ravel()
test_y = test_df[Y_columns].values.ravel()

# take Y_columns as the label, and transfering to one-hot coded
dataset = {
    'train_input': torch.tensor(train_X_scaled, dtype=torch.float32, device=device),
    'train_label': F.one_hot(torch.tensor(train_y, dtype=torch.long, device=device), num_classes=4),
    'test_input': torch.tensor(test_X_scaled, dtype=torch.float32, device=device),
    'test_label': F.one_hot(torch.tensor(test_y, dtype=torch.long, device=device), num_classes=4)
}
print("Data prepared.",
      f"Train set: {dataset['train_input'].shape, dataset['train_label'].shape}",
      f"Test set: {dataset['test_input'].shape, dataset['test_label'].shape}", sep="\n")

cpu
Data prepared.
Train set: (torch.Size([33048, 14]), torch.Size([33048, 4]))
Test set: (torch.Size([3672, 14]), torch.Size([3672, 4]))


#### Downsampling to get Medium size data  
Large: filtered_train_l_4_11.csv with shape (99909, 47)  
Medium: filtered_train_m_4_11.csv with shape (66000, 47)  
Small: filtered_train_s_4_11.csv with shape (36720, 47)  

For test sets, we have two available:  
the large one 'filtered_test_4_11.csv' with shape (11101, 47)  
the small one 'logiKNet_test_3994.csv', sampled from the Small dataset.

In [ ]:
# Downsample large 4_11 dataset to a medium-sized training set
from pathlib import Path

# Source large dataset and output path
src_path = Path('/home/zyang44/Github/baseline_cicIOT/CIC_IoMT/19classes/filtered_train_l_4_11.csv')
out_path = Path('/home/zyang44/Github/baseline_cicIOT/CIC_IoMT/19classes/filtered_train_m_4_11.csv')

# Target medium counts per Label_L1 (sum = 66,000)
target_counts_l1 = {
    'MQTT': 30000,
    'Benign': 18000,
    'Recon': 12000,
    'ARP_Spoofing': 6000,
}

# Load large dataset
df_l = pd.read_csv(src_path)
print('Loaded large dataset:', src_path)
print('Large shape:', df_l.shape)

# Verify available counts and sample per class
available_counts = df_l['label_L1'].value_counts().to_dict()
print('Available Label_L1 counts:', available_counts)

dfs = []
rng = 42
for label, target in target_counts_l1.items():
    group = df_l[df_l['label_L1'] == label]
    if group.empty:
        print(f'Warning: no rows for Label_L1 {label}; skipping.')
        continue
    n = min(target, len(group))
    sampled = group.sample(n=n, random_state=rng)
    dfs.append(sampled)
    print(f'Sampled Label_L1 {label}: {n} rows')

# Concatenate and shuffle
df_m = pd.concat(dfs, axis=0).sample(frac=1, random_state=rng).reset_index(drop=True)
print('Custom Reduced Data Shape:', df_m.shape)

# Print summary counts for label_L1 and label_L2
print('label_L1 Counts:')
for lbl, cnt in df_m['label_L1'].value_counts().to_dict().items():
    print(f'Label_L1 {lbl}: {cnt} entries')

if 'label_L2' in df_m.columns:
    print('\nlabel_L2 Counts:')
    for lbl, cnt in df_m['label_L2'].value_counts().to_dict().items():
        print(f'Label_L2 {lbl}: {cnt} entries')

# Save medium dataset
df_m.to_csv(out_path, index=False)
print(f'Training data saved to {out_path} with shape {df_m.shape}')

Loaded large dataset: /home/zyang44/Github/baseline_cicIOT/CIC_IoMT/19classes/filtered_train_l_4_11.csv
Large shape: (99909, 47)
Available Label_L1 counts: {'MQTT': 42189, 'Benign': 27000, 'Recon': 21720, 'ARP_Spoofing': 9000}
Sampled Label_L1 MQTT: 30000 rows
Sampled Label_L1 Benign: 18000 rows
Sampled Label_L1 Recon: 12000 rows
Sampled Label_L1 ARP_Spoofing: 6000 rows
Custom Reduced Data Shape: (66000, 47)
label_L1 Counts:
Label_L1 MQTT: 30000 entries
Label_L1 Benign: 18000 entries
Label_L1 Recon: 12000 entries
Label_L1 ARP_Spoofing: 6000 entries

label_L2 Counts:
Label_L2 benign: 18000 entries
Label_L2 MQTT-DoS-Publish_Flood: 6424 entries
Label_L2 MQTT-DDoS-Publish_Flood: 6411 entries
Label_L2 MQTT-DDoS-Connect_Flood: 6406 entries
Label_L2 MQTT-DoS-Connect_Flood: 6338 entries
Label_L2 arp_spoofing: 6000 entries
Label_L2 Recon-OS_Scan: 5000 entries
Label_L2 Recon-Port_Scan: 4933 entries
Label_L2 MQTT-Malformed_Data: 4421 entries
Label_L2 Recon-VulScan: 1614 entries
Label_L2 Recon-Pin

#### Save datasets (after P2, 14 features left)
'logiKNet_test_3994.csv' might be included in Large/Medium datasets. NOT WORK.  
Scaling test set of 'filtered_test_4_11' by training set.

In [ ]:
X_columns = [
    'Header_Length', 
    # 'Protocol Type', 'Duration', # Second Pruned (16->14)
    'Rate', 'Srate', 
    # 'Drate',
    # 'fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number',
    # 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number', 'ack_count',
    # 'syn_count', 'fin_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet',
    # 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IGMP', 
    # 'IPv','LLC',  # First Pruned (18->16)
    'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number',
    'Magnitue', 'Radius', 'Covariance',
    # 'Variance', 'Weight'
]

Y_columns = ['label_L1']

import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Source training files
org_large = '/home/zyang44/Github/baseline_cicIOT/CIC_IoMT/19classes/filtered_train_l_4_11.csv'
org_medium = '/home/zyang44/Github/baseline_cicIOT/CIC_IoMT/19classes/filtered_train_m_4_11.csv'
org_small = '/home/zyang44/Github/baseline_cicIOT/CIC_IoMT/19classes/filtered_train_s_4_11.csv'

# Common raw test file to reuse for all experiments
common_test_src = '/home/zyang44/Github/baseline_cicIOT/CIC_IoMT/19classes/filtered_test_4_11.csv'

# Ensure output dir exists
out_dir = Path('./input_files')
out_dir.mkdir(parents=True, exist_ok=True)

# Label mappings (used only if labels are strings)
label_L1_mapping = {"MQTT": 0, "Benign": 1, "Recon": 2, "ARP_Spoofing": 3}
label_L2_mapping = {
    "MQTT-DDoS-Connect_Flood": 0, "MQTT-DDoS-Publish_Flood": 1,
    "MQTT-DoS-Connect_Flood": 2, "MQTT-DoS-Publish_Flood": 3,
    "MQTT-Malformed_Data": 4, "benign": 5, "Benign": 5,
    "Recon-OS_Scan": 10, "Recon-Port_Scan": 11, "arp_spoofing": 12
}

def map_label(series: pd.Series, col_name: str) -> pd.Series:
    if np.issubdtype(series.dtype, np.number):
        return series.astype(int)
    if col_name == 'label_L1':
        return series.map(label_L1_mapping).astype(int)
    elif col_name == 'label_L2':
        return series.map(label_L2_mapping).astype(int)
    # Fallback
    codes, _ = pd.factorize(series)
    return pd.Series(codes, index=series.index).astype(int)

y_col = Y_columns[0]  # use provided Y_columns

sources = {
    "large": org_large,
    "medium": org_medium,
    "small": org_small,
}

# Load common test (raw), select features and label
df_test_raw = pd.read_csv(common_test_src)
X_test_raw = df_test_raw[X_columns].copy()
y_test_raw = map_label(df_test_raw[y_col], y_col)

for size, src in sources.items():
    # Train: read, select, scale, save
    df_train = pd.read_csv(src)
    X_train = df_train[X_columns].copy()
    y_train = map_label(df_train[y_col], y_col)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    df_train_out = pd.DataFrame(X_train_scaled, columns=X_columns)
    df_train_out[y_col] = y_train.values
    train_out_path = out_dir / f'logiKNet_train_{size}.csv'
    df_train_out.to_csv(train_out_path, index=False)
    print(f'Saved train {size}: {train_out_path} with shape {df_train_out.shape}')

    # Test: transform using the scaler fitted on this train
    X_test_scaled = scaler.transform(X_test_raw)
    df_test_out = pd.DataFrame(X_test_scaled, columns=X_columns)
    df_test_out[y_col] = y_test_raw.values
    test_out_path = out_dir / f'logiKNet_test_scaled_by_{size}.csv'
    df_test_out.to_csv(test_out_path, index=False)
    print(f'Saved test (common, scaled by {size}): {test_out_path} with shape {df_test_out.shape}')

Saved train large: input_files/logiKNet_train_large.csv with shape (99909, 15)
Saved test (common, scaled by large): input_files/logiKNet_test_scaled_by_large.csv with shape (11101, 15)
Saved train medium: input_files/logiKNet_train_medium.csv with shape (66000, 15)
Saved test (common, scaled by medium): input_files/logiKNet_test_scaled_by_medium.csv with shape (11101, 15)
Saved train small: input_files/logiKNet_train_small.csv with shape (36720, 15)
Saved test (common, scaled by small): input_files/logiKNet_test_scaled_by_small.csv with shape (11101, 15)


#### Save Models - MLP/ Logic-MLP/ LogiK-Net

In [6]:
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
import os

##### Utils

In [15]:
def load_csv_data(input_folder: str,
                  train_fname: str,
                  test_fname: str):
    """
    Reads train & test CSVs from disk.
    
    Returns:
      train_df, test_df (both pandas.DataFrame)
    """
    train_path = os.path.join(input_folder, train_fname)
    test_path  = os.path.join(input_folder, test_fname)
    train_df = pd.read_csv(train_path)
    test_df  = pd.read_csv(test_path)
    return train_df, test_df

def extract_features_labels(df: pd.DataFrame):
    """
    Splits a DataFrame into numpy feature array X and label vector y.
    
    The last column is the label.
    """
    X = df.iloc[:, :-1].values
    y = df.iloc[:,  -1].values
    return X, y

# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load data
input_folder = './input_files'
train_fname = 'logiKNet_train_large.csv'
test_fname = 'logiKNet_test_scaled_by_large.csv'

train_df, test_df = load_csv_data(input_folder, train_fname, test_fname)
# Extract features and labels   
X_train, y_train = extract_features_labels(train_df)
X_test, y_test = extract_features_labels(test_df)

# dataset = {
#     'train_input': torch.tensor(X_train, dtype=torch.float32, device=device),
#     'train_label': F.one_hot(torch.tensor(y_train, dtype=torch.long, device=device), num_classes=6),
#     'test_input': torch.tensor(X_test, dtype=torch.float32, device=device),
#     'test_label': F.one_hot(torch.tensor(y_test, dtype=torch.long, device=device), num_classes=6)
# }

dataset_numeric = {
    'train_input': torch.tensor(X_train, dtype=torch.float32, device=device),
    'train_label': torch.tensor(y_train, dtype=torch.long, device=device),
    'test_input': torch.tensor(X_test, dtype=torch.float32, device=device),
    'test_label': torch.tensor(y_test, dtype=torch.long, device=device)
}

# this is a standard PyTorch DataLoader to load the dataset for the training and testing of the model
class DataLoader(object):
    def __init__(self,
                 data,
                 labels,
                 batch_size=1,
                 shuffle=True):
        self.data = data
        self.labels = labels
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __len__(self):
        return int(np.ceil(self.data.shape[0] / self.batch_size))

    def __iter__(self):
        n = self.data.shape[0]
        idxlist = list(range(n))
        if self.shuffle:
            np.random.shuffle(idxlist)

        for _, start_idx in enumerate(range(0, n, self.batch_size)):
            end_idx = min(start_idx + self.batch_size, n)
            data = self.data[idxlist[start_idx:end_idx]]
            labels = self.labels[idxlist[start_idx:end_idx]]
            ############################################################
            # Check if any class is missing in the batch
            # present_classes = np.unique(labels.cpu().numpy())
            # all_classes = np.arange(len(label_mapping))  # Adjust based on number of classes
            # missing_classes = set(all_classes) - set(present_classes)
            #
            # if missing_classes:
            #     print(f"Batch {start_idx // self.batch_size} is missing classes {missing_classes}")
            ############################################################
            yield data, labels


train_loader = DataLoader(
    dataset_numeric['train_input'],
    dataset_numeric['train_label'], 
    batch_size=len(X_train), 
    shuffle=True
    )
test_loader = DataLoader(
    dataset_numeric['test_input'],
    dataset_numeric['test_label'],
    batch_size=len(X_test), 
    # batch_size=1,
    shuffle=False
    )

In [16]:
class LogitsToPredicate(torch.nn.Module):
    """
    This model has inside a logits model, that is a model which compute logits for the classes given an input example x.
    The idea of this model is to keep logits and probabilities separated. The logits model returns the logits for an example,
    while this model returns the probabilities given the logits model.

    In particular, it takes as input an example x and a class label l. It applies the logits model to x to get the logits.
    Then, it applies a softmax function to get the probabilities per classes. Finally, it returns only the probability related
    to the given class l.
    """

    def __init__(self, logits_model):
        super(LogitsToPredicate, self).__init__()
        self.logits_model = logits_model
        self.softmax = torch.nn.Softmax(dim=1)

    def forward(self, x, l, training=False):
        logits = self.logits_model(x, training=training)
        probs = self.softmax(logits)
        out = torch.sum(probs * l, dim=1)  # 计算并返回与给定类标签l对应的概率值
        return out


class MLP(torch.nn.Module):
    """
    This model returns the logits for the classes given an input example. It does not compute the softmax, so the output
    are not normalized.
    This is done to separate the accuracy computation from the satisfaction level computation. Go through the example
    to understand it.
    """

    def __init__(self, layer_sizes):
        super(MLP, self).__init__()
        self.elu = torch.nn.ELU()
        self.dropout = torch.nn.Dropout(0.2)
        self.linear_layers = torch.nn.ModuleList([torch.nn.Linear(layer_sizes[i - 1], layer_sizes[i])
                                                  for i in range(1, len(layer_sizes))])

    def forward(self, x, training=False):
        """
        Method which defines the forward phase of the neural network for our multi class classification task.
        In particular, it returns the logits for the classes given an input example.

        :param x: the features of the example
        :param training: whether the network is in training mode (dropout applied) or validation mode (dropout not applied)
        :return: logits for example x
        """
        for layer in self.linear_layers[:-1]:
            x = self.elu(layer(x))
            if training:
                x = self.dropout(x)
        logits = self.linear_layers[-1](x)
        return logits


class MultiKANModel(torch.nn.Module):
    def __init__(self, kan):
        """
        Wrap an already built MultKAN instance.
        Args:
            kan: a MultKAN model (which has attributes such as act_fun, symbolic_fun, node_bias, node_scale,
                 subnode_bias, subnode_scale, depth, width, mult_homo, mult_arity, input_id, symbolic_enabled, etc.)
        """
        super(MultiKANModel, self).__init__()
        self.kan = kan

    def forward(self, x, training=False, singularity_avoiding=False, y_th=10.):
        # Select input features according to input_id
        x = x[:, self.kan.input_id.long()]
        # Loop through each layer
        for l in range(self.kan.depth):
            # Get outputs from the numerical branch (KANLayer) of current layer
            x_numerical, preacts, postacts_numerical, postspline = self.kan.act_fun[l](x)
            # Get output from the symbolic branch if enabled
            if self.kan.symbolic_enabled:
                x_symbolic, postacts_symbolic = self.kan.symbolic_fun[l](x, singularity_avoiding=singularity_avoiding, y_th=y_th)
            else:
                x_symbolic = 0.
            # Sum the numerical and symbolic outputs
            x = x_numerical + x_symbolic

            # Subnode affine transformation
            x = self.kan.subnode_scale[l][None, :] * x + self.kan.subnode_bias[l][None, :]

            # Process multiplication nodes
            dim_sum = self.kan.width[l+1][0]
            dim_mult = self.kan.width[l+1][1]
            if dim_mult > 0:
                if self.kan.mult_homo:
                    for i in range(self.kan.mult_arity-1):
                        if i == 0:
                            x_mult = x[:, dim_sum::self.kan.mult_arity] * x[:, dim_sum+1::self.kan.mult_arity]
                        else:
                            x_mult = x_mult * x[:, dim_sum+i+1::self.kan.mult_arity]
                else:
                    for j in range(dim_mult):
                        acml_id = dim_sum + int(np.sum(self.kan.mult_arity[l+1][:j]))
                        for i in range(self.kan.mult_arity[l+1][j]-1):
                            if i == 0:
                                x_mult_j = x[:, [acml_id]] * x[:, [acml_id+1]]
                            else:
                                x_mult_j = x_mult_j * x[:, [acml_id+i+1]]
                        if j == 0:
                            x_mult = x_mult_j
                        else:
                            x_mult = torch.cat([x_mult, x_mult_j], dim=1)
                # Concatenate sum and mult parts
                x = torch.cat([x[:, :dim_sum], x_mult], dim=1)

            # Node affine transformation
            x = self.kan.node_scale[l][None, :] * x + self.kan.node_bias[l][None, :]

        # Final x corresponds to the logits output of the whole model
        return x

In [17]:
def compute_accuracy(loader, model):
    total_correct = 0
    total_samples = 0
    for data, labels in loader:
        logits = model(data)
        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == labels).sum()
        total_samples += labels.numel()
    return total_correct.float() / total_samples


def save_model(model, model_save_folder, model_name):
    """
    Save the model to disk.
    """
    torch.save(model.state_dict(), os.path.join(model_save_folder, model_name))

    print(f"Model saved to {os.path.join(model_save_folder, model_name)}")


model_state_folder = './model_weights'

##### MLP 

In [12]:
#  Define the MLP predicate
mlp = MLP(layer_sizes=(14, 10, 4)).to(device)

# MLP with standard loss fn 
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=0.001)

# Train the model
for epoch in range(401):
    for data, labels in train_loader:               # iterate batches
        optimizer.zero_grad()
        logits = mlp(data, training=True)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
    # test
    acc = compute_accuracy(test_loader, mlp)
    print(f"Epoch {epoch}, Loss: {loss.item()}, Test accuracy: {acc.item()}")

# Save the model
save_model(mlp, model_state_folder, 'mlp_L.pt')

Epoch 0, Loss: 1.4488524198532104, Test accuracy: 0.2288983017206192
Epoch 1, Loss: 1.4450145959854126, Test accuracy: 0.23042969405651093
Epoch 2, Loss: 1.4414656162261963, Test accuracy: 0.23196108639240265
Epoch 3, Loss: 1.4380383491516113, Test accuracy: 0.23412305116653442
Epoch 4, Loss: 1.4351418018341064, Test accuracy: 0.23673543334007263
Epoch 5, Loss: 1.4311622381210327, Test accuracy: 0.2400684654712677
Epoch 6, Loss: 1.4284322261810303, Test accuracy: 0.2436717450618744
Epoch 7, Loss: 1.425262451171875, Test accuracy: 0.24772542715072632
Epoch 8, Loss: 1.4220610857009888, Test accuracy: 0.25195929408073425
Epoch 9, Loss: 1.4193838834762573, Test accuracy: 0.25691378116607666
Epoch 10, Loss: 1.415642499923706, Test accuracy: 0.25952616333961487
Epoch 11, Loss: 1.4136086702346802, Test accuracy: 0.26195839047431946
Epoch 12, Loss: 1.4106028079986572, Test accuracy: 0.2652914226055145
Epoch 13, Loss: 1.4074598550796509, Test accuracy: 0.2671831250190735
Epoch 14, Loss: 1.40500

In [ ]:
def load_model_state(infer_model, model_save_folder, model_name):
    """
    Load the model from disk.
    """
    checkpoint = torch.load(os.path.join(model_save_folder, model_name), 
                            map_location=device)
    infer_model.load_state_dict(checkpoint)
    infer_model.eval()
    return infer_model

# Load the model
mlp_infer = MLP(layer_sizes=(14, 10, 401)).to(device)
mlp_infer = load_model_state(mlp_infer, model_state_folder, 'mlp_L.pt')
# Test the model
with torch.no_grad():
    for data, labels in test_loader:
        logits = mlp_infer(data)
        preds = torch.argmax(logits, dim=1)
        print(f"Predictions: {preds}, Labels: {labels}")
        
        mlp_infer_acc = compute_accuracy(test_loader, mlp_infer)
        print(f"Test accuracy of the loaded model: {mlp_infer_acc.item()}")
        break

/tmp/ipykernel_2201336/3224332884.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(os.path.join(model_save_folder, model_name),


Predictions: tensor([2]), Labels: tensor([2])
Test accuracy of the loaded model: 0.7873164415359497


##### LTN setup

In [18]:
import ltn
import ltn.fuzzy_ops
import torch
import os
from kan import KAN

# define the connectives, quantifiers, and the SatAgg
Not = ltn.Connective(ltn.fuzzy_ops.NotStandard())
And = ltn.Connective(ltn.fuzzy_ops.AndProd())   # And = ltn.Connective(custom_fuzzy_ops.AndProd())
Or = ltn.Connective(ltn.fuzzy_ops.OrProbSum())
Forall = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")
Exists = ltn.Quantifier(ltn.fuzzy_ops.AggregPMean(p=2), quantifier="e")
Implies = ltn.Connective(ltn.fuzzy_ops.ImpliesReichenbach())
SatAgg = ltn.fuzzy_ops.SatAgg()

# define ltn constants
l_MQTT = ltn.Constant(torch.tensor([1, 0, 0, 0]))
l_Benign = ltn.Constant(torch.tensor([0, 1, 0, 0]))
l_Recon = ltn.Constant(torch.tensor([0, 0, 1, 0]))
l_ARP_Spoofing = ltn.Constant(torch.tensor([0, 0, 0, 1]))

##### Logic-MLP

In [19]:
def compute_sat_levels(loader, P):
	sat_level  = 0
	for data, labels in loader:
		x = ltn.Variable("x", data)
		x_MQTT = ltn.Variable("x_MQTT", data[labels == 0])
		x_Benign = ltn.Variable("x_Benign", data[labels == 1])
		x_Recon = ltn.Variable("x_Recon", data[labels == 2])
		x_ARP_Spoofing = ltn.Variable("x_ARP_Spoofing", data[labels == 3])
		
		sat_level = SatAgg(
			Forall(x_MQTT, P(x_MQTT, l_MQTT)),
			Forall(x_Benign, P(x_Benign, l_Benign)),
			Forall(x_Recon, P(x_Recon, l_Recon)),
			Forall(x_ARP_Spoofing, P(x_ARP_Spoofing, l_ARP_Spoofing))
		)
	return sat_level

mlp = MLP(layer_sizes=(14, 10, 4)).to(device)
P_mlp = ltn.Predicate(LogitsToPredicate(mlp))

optimizer = torch.optim.Adam(P_mlp.parameters(), lr=0.001)

# Train the model
for epoch in range(401):             
	optimizer.zero_grad()
	sat_mlp = compute_sat_levels(train_loader, P_mlp)
	loss = 1. - sat_mlp
	loss.backward()
	optimizer.step()
	train_loss_mlp  = loss.item()
	
	# test
	test_acc_mlp = compute_accuracy(test_loader, mlp)
	test_sat_mlp = compute_sat_levels(test_loader, P_mlp)
	print(f"Epoch {epoch} | Logic-MLP (loss/acc/sat): {train_loss_mlp:.3f}/{test_acc_mlp:.3f}/{sat_mlp:.3f}({test_sat_mlp:.3f})")

print("Training finished.")

# save the models as .pt files
save_model(mlp, model_state_folder, 'logic_mlp.pt')

Epoch 0 | Logic-MLP (loss/acc/sat): 0.745/0.329/0.255(0.257)
Epoch 1 | Logic-MLP (loss/acc/sat): 0.744/0.330/0.256(0.258)
Epoch 2 | Logic-MLP (loss/acc/sat): 0.742/0.330/0.258(0.259)
Epoch 3 | Logic-MLP (loss/acc/sat): 0.741/0.331/0.259(0.260)
Epoch 4 | Logic-MLP (loss/acc/sat): 0.740/0.332/0.260(0.261)
Epoch 5 | Logic-MLP (loss/acc/sat): 0.739/0.333/0.261(0.262)
Epoch 6 | Logic-MLP (loss/acc/sat): 0.738/0.334/0.262(0.263)
Epoch 7 | Logic-MLP (loss/acc/sat): 0.737/0.335/0.263(0.264)
Epoch 8 | Logic-MLP (loss/acc/sat): 0.736/0.335/0.264(0.265)
Epoch 9 | Logic-MLP (loss/acc/sat): 0.735/0.335/0.265(0.267)
Epoch 10 | Logic-MLP (loss/acc/sat): 0.734/0.334/0.266(0.268)
Epoch 11 | Logic-MLP (loss/acc/sat): 0.733/0.335/0.267(0.269)
Epoch 12 | Logic-MLP (loss/acc/sat): 0.732/0.335/0.268(0.270)
Epoch 13 | Logic-MLP (loss/acc/sat): 0.731/0.335/0.269(0.271)
Epoch 14 | Logic-MLP (loss/acc/sat): 0.730/0.335/0.270(0.272)
Epoch 15 | Logic-MLP (loss/acc/sat): 0.729/0.336/0.271(0.273)
Epoch 16 | Logic-M

##### LogiK-Net

In [ ]:
def compute_sat_levels(loader, P):
	sat_level  = 0
	for data, labels in loader:
		x = ltn.Variable("x", data)
		x_MQTT = ltn.Variable("x_MQTT", data[labels == 0])
		x_Benign = ltn.Variable("x_Benign", data[labels == 1])
		x_Recon = ltn.Variable("x_Recon", data[labels == 2])
		x_ARP_Spoofing = ltn.Variable("x_ARP_Spoofing", data[labels == 3])
		
		sat_level = SatAgg(
			Forall(x_MQTT, P(x_MQTT, l_MQTT)),
			Forall(x_Benign, P(x_Benign, l_Benign)),
			Forall(x_Recon, P(x_Recon, l_Recon)),
			Forall(x_ARP_Spoofing, P(x_ARP_Spoofing, l_ARP_Spoofing))
		)
	return sat_level


kan = KAN(width=[14, 10, 4], grid=5, k=3, seed=42, device=device)
P_kan = ltn.Predicate(LogitsToPredicate(MultiKANModel(kan)))

optimizer_kan = torch.optim.Adam(P_kan.parameters(), lr=0.001)

for epoch in range(401):
	optimizer_kan.zero_grad()
	sat_kan = compute_sat_levels(train_loader, P_kan)
	loss = 1. - sat_kan
	loss.backward()
	optimizer_kan.step()
	train_loss_kan = loss.item()

	# Test the KAN
	test_acc_kan = compute_accuracy(test_loader, kan)
	test_sat_kan = compute_sat_levels(test_loader, P_kan)

	print(f"Epoch {epoch} | KAN (loss/acc/sat): {train_loss_kan:.3f}/{test_acc_kan:.3f}/{sat_kan:.3f}({test_sat_kan:.3f})")

print("Training finished.")

# save the models as .pt files
save_model(kan, model_state_folder, 'logiKNet.pt')

checkpoint directory created: ./model
saving model version 0.0
Epoch 0 | KAN (loss/acc/sat): 0.746/0.263/0.254(0.255)
Epoch 1 | KAN (loss/acc/sat): 0.745/0.265/0.255(0.256)
Epoch 2 | KAN (loss/acc/sat): 0.744/0.266/0.256(0.257)
Epoch 3 | KAN (loss/acc/sat): 0.743/0.268/0.257(0.258)
Epoch 4 | KAN (loss/acc/sat): 0.742/0.269/0.258(0.259)
Epoch 5 | KAN (loss/acc/sat): 0.741/0.271/0.259(0.260)
Epoch 6 | KAN (loss/acc/sat): 0.740/0.274/0.260(0.260)
Epoch 7 | KAN (loss/acc/sat): 0.740/0.276/0.260(0.261)
Epoch 8 | KAN (loss/acc/sat): 0.739/0.277/0.261(0.262)
Epoch 9 | KAN (loss/acc/sat): 0.738/0.278/0.262(0.263)
Epoch 10 | KAN (loss/acc/sat): 0.737/0.279/0.263(0.264)
Epoch 11 | KAN (loss/acc/sat): 0.736/0.279/0.264(0.265)
Epoch 12 | KAN (loss/acc/sat): 0.735/0.280/0.265(0.265)
Epoch 13 | KAN (loss/acc/sat): 0.735/0.280/0.265(0.266)
Epoch 14 | KAN (loss/acc/sat): 0.734/0.281/0.266(0.267)
Epoch 15 | KAN (loss/acc/sat): 0.733/0.281/0.267(0.268)
Epoch 16 | KAN (loss/acc/sat): 0.732/0.281/0.268(0.